# Week 3: Imbalanced Classification and LightGBM Modeling

**Day 1 - Confirm the imbalance problem**

Goal for today: load the fused dataset from Week 2, measure exactly how rare machine failures are, install the libraries you'll need for the rest of the week, and see concretely why accuracy is the wrong metric for this problem before writing a single line of modeling code.



In [1]:
import pandas as pd
import numpy as np

import lightgbm as lgb
import imblearn

print("lightgbm:", lgb.__version__)
print("imbalanced-learn:", imblearn.__version__)

lightgbm: 4.6.0
imbalanced-learn: 0.14.2


In [2]:
DATA_PATH = "ai4i_fused_week2.csv" 

df = pd.read_csv(DATA_PATH)
print(df.shape)
df.head()

(10000, 22)


,UDI,Product ID,Type,Air temperature [K],Process temperature [K],Rotational speed [rpm],Torque [Nm],Tool wear [min],Machine failure,TWF,...,OSF,RNF,timestamp,ambient_temp_C,load_density_pct,ambient_temp_K,temp_differential_K,load_adjusted_torque,wear_per_load,Type_enc
0,1,M14860,M,298.1,308.6,1551,42.8,0,0,0,...,0,0,1/1/2024 0:00,18.913477,38.362480,292.063477,6.036523,108.732986,0.000000,2
1,2,L47181,L,298.2,308.7,1408,46.3,3,0,0,...,0,0,1/1/2024 0:10,18.913477,38.362480,292.063477,6.136523,117.624702,1.150874,1
2,3,L47182,L,298.1,308.5,1498,49.4,5,0,0,...,0,0,1/1/2024 0:20,18.913477,38.362480,292.063477,6.036523,125.500222,1.918124,1
3,4,L47183,L,298.2,308.6,1433,39.5,7,0,0,...,0,0,1/1/2024 0:30,18.913477,38.362480,292.063477,6.136523,100.349368,2.685374,1
4,5,L47184,L,298.2,308.7,1408,40.0,9,0,0,...,0,0,1/1/2024 0:40,17.610902,37.186934,290.760902,7.439098,104.747869,3.346824,1


## Step 1 - Measure exactly how rare failures are

The brief states failures are under 2% of the data - let's verify that on your actual fused dataset rather than assuming.

In [3]:
target = "Machine failure"

counts = df[target].value_counts()
pct = df[target].value_counts(normalize=True) * 100

print(counts)
print()
print(pct.round(3))

Machine failure
0    9661
1     339
Name: count, dtype: int64

Machine failure
0    96.61
1     3.39
Name: proportion, dtype: float64


## Step 2 - Why accuracy will lie to you

Before touching LightGBM, let's prove the point with the simplest possible "model": one that always predicts "no failure," no matter what. If this dummy model still scores high on accuracy, you know accuracy can't be trusted to judge anything for the rest of this week.

In [4]:
from sklearn.dummy import DummyClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, recall_score, f1_score

y = df[target]

y_train, y_test = train_test_split(y, test_size=0.2, stratify=y, random_state=42)

dummy = DummyClassifier(strategy="most_frequent")
dummy.fit(np.zeros((len(y_train), 1)), y_train)
y_pred = dummy.predict(np.zeros((len(y_test), 1)))

print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}")
print(f"Recall (failures caught): {recall_score(y_test, y_pred):.4f}")
print(f"F1 score: {f1_score(y_test, y_pred):.4f}")

Accuracy: 0.9660
Recall (failures caught): 0.0000
F1 score: 0.0000


## Takeaway for Day 1

A model that never predicts a failure can score over 95% accuracy while catching **zero** real failures - exactly the opposite of what a predictive maintenance system needs to do. For the rest of this week, judge every model by **recall**, **F1**, and **ROC-AUC / PR-AUC**, never accuracy alone.



**Day 2 - Build the real 5-fold stratified CV pipeline (no SMOTE yet)**

Goal: set up `StratifiedKFold`, train a baseline LightGBM with no resampling tricks, and record real recall/F1/ROC-AUC/PR-AUC numbers across the 5 folds. 

### Step 0 - Watch out for a built-in leak

In the AI4I dataset, `Machine failure` is literally defined as 1 whenever *any* of the five failure-mode flags (`TWF`, `HDF`, `PWF`, `OSF`, `RNF`) is 1. If you leave those columns in as features, your model will "predict" failure by just reading the answer off the failure-mode columns - you'd get near-perfect recall and it would mean nothing. Drop them, along with ID/text columns that aren't real signal.

In [5]:
target = "Machine failure"
leak_cols = ["TWF", "HDF", "PWF", "OSF", "RNF"]
drop_cols = ["UDI", "Product ID", "Type", "timestamp", target] + leak_cols

feature_cols = [c for c in df.columns if c not in drop_cols]
print(f"Using {len(feature_cols)} features:")
print(feature_cols)

X = df[feature_cols]
y = df[target]

Using 12 features:
['Air temperature [K]', 'Process temperature [K]', 'Rotational speed [rpm]', 'Torque [Nm]', 'Tool wear [min]', 'ambient_temp_C', 'load_density_pct', 'ambient_temp_K', 'temp_differential_K', 'load_adjusted_torque', 'wear_per_load', 'Type_enc']


### Step 1 - Why stratified matters here

With failures under 2% of rows, a plain random K-fold could easily place very few (or even zero) failures into a given fold by chance, making that fold's metrics meaningless. `StratifiedKFold` forces each fold to keep roughly the same failure rate as the full dataset. Quick check before training anything:

In [6]:
from sklearn.model_selection import StratifiedKFold

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

for i, (train_idx, test_idx) in enumerate(cv.split(X, y), start=1):
    fold_rate = y.iloc[test_idx].mean() * 100
    print(f"Fold {i}: {len(test_idx)} rows, failure rate {fold_rate:.3f}%")

Fold 1: 2000 rows, failure rate 3.350%
Fold 2: 2000 rows, failure rate 3.400%
Fold 3: 2000 rows, failure rate 3.400%
Fold 4: 2000 rows, failure rate 3.400%
Fold 5: 2000 rows, failure rate 3.400%


### Step 2 - Train a plain LightGBM per fold (the baseline)

No SMOTE, no class weighting - just LightGBM as-is.

In [7]:
import lightgbm as lgb
from sklearn.metrics import (
    recall_score, precision_score, f1_score,
    roc_auc_score, average_precision_score
)

def sanitize_col_names(df):
    # Replace problematic characters with underscores
    new_columns = []
    for col in df.columns:
        col = col.replace('[', '_').replace(']', '_').replace('<', '_').replace('>', '_').replace(' ', '_').replace('=', '_').replace(',', '_')
        new_columns.append(col)
    df.columns = new_columns
    return df

def run_cv(X, y, cv, model_fn):
    rows = []
    for train_idx, test_idx in cv.split(X, y):
        X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
        y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

        # Sanitize column names for LightGBM
        X_train = sanitize_col_names(X_train.copy())
        X_test = sanitize_col_names(X_test.copy())

        model = model_fn()
        model.fit(X_train, y_train)

        proba = model.predict_proba(X_test)[:, 1]
        pred = model.predict(X_test)

        rows.append({
            "recall": recall_score(y_test, pred),
            "precision": precision_score(y_test, pred, zero_division=0),
            "f1": f1_score(y_test, pred),
            "roc_auc": roc_auc_score(y_test, proba),
            "pr_auc": average_precision_score(y_test, proba),
        })
    return pd.DataFrame(rows)

baseline_model_fn = lambda: lgb.LGBMClassifier(random_state=42, verbose=-1)

baseline_results = run_cv(X, y, cv, baseline_model_fn)
baseline_results

,recall,precision,f1,roc_auc,pr_auc
0,0.671642,0.882353,0.762712,0.981955,0.839885
1,0.705882,0.888889,0.786885,0.980514,0.849772
2,0.573529,0.847826,0.684211,0.983703,0.779980
3,0.735294,0.819672,0.775194,0.978421,0.835355
4,0.544118,0.880952,0.672727,0.966592,0.784036


In [8]:
summary = baseline_results.agg(["mean", "std"]).T
summary.columns = ["mean", "std"]
print("Baseline LightGBM (no resampling) - 5-fold CV results:")
summary.round(4)

Baseline LightGBM (no resampling) - 5-fold CV results:


,mean,std
recall,0.6461,0.0834
precision,0.8639,0.0295
f1,0.7363,0.0537
roc_auc,0.9782,0.0068
pr_auc,0.8178,0.0331


In [9]:
baseline_results.to_csv("week3_baseline_cv_results.csv", index=False)
print("Saved week3_baseline_cv_results.csv")

Saved week3_baseline_cv_results.csv


## Takeaway for Day 2

we now have a real baseline: a plain LightGBM, evaluated honestly across 5 stratified folds, with leakage columns removed. Whatever recall/F1/PR-AUC numbers you got above are the bar Day 3's SMOTE pipeline has to clear - and clear by a real margin, since SMOTE adds its own noise too.

